# Cost-aware Multi-Fidelity Bayesian Optimization

このNotebookでは、同じ評価コスト予算の下で、BoTorch標準のMF-KGとランダム探索を比較します。
評価fidelityとtarget fidelityを区別し、予測費用の予約・実測費用の精算を記録します。

**注意:** target-fidelity oracleは合成関数を用いたベンチマーク診断用です。学習には渡さず、実運用で無料の評価を仮定しません。

In [ ]:
from pathlib import Path
import importlib.util
import pandas as pd
import matplotlib.pyplot as plt

benchmark = Path.cwd() / 'benchmarks' / 'multifidelity_cost_budget.py'
if not benchmark.exists():
    benchmark = Path.cwd().parent.parent / 'benchmarks' / 'multifidelity_cost_budget.py'
spec = importlib.util.spec_from_file_location('mf_cost_benchmark', benchmark)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)


In [ ]:
records = [row for seed in (0, 1) for policy in ('random', 'mfkg')
           for row in module.experiment(policy, seed, total_budget=3.0)]
results = pd.DataFrame(records)
display(results)


In [ ]:
fig, ax = plt.subplots()
for (policy, seed), group in results.groupby(['policy', 'seed']):
    ax.plot(group['spent'], group['best_target_oracle'], marker='o', label=f'{policy}, seed={seed}')
ax.set(xlabel='Cumulative evaluation cost', ylabel='Best target-fidelity oracle value')
ax.legend()
plt.show()


## 解釈上の注意

初期観測は両手法で共通であり、その費用は比較予算に含めません。候補選択に必要な計算時間も費用に含めていません。ランダム探索は連続fidelityから候補を選びます。target-fidelity oracleは合成関数でしか計算できない診断量です。少数seedの結果から優越性を結論づけないでください。
